# Lab: a recoverable in-memory report queue
Deterministic simulation only: no sleeps, network, or external side effects.


In [ ]:
from dataclasses import dataclass, field
print('Python environment ready')


## Objectives
Model submit/worker states, test accepted versus completed, reconcile the outbox, handle duplicate delivery idempotently, and bound retries/dead letters.


## Baseline reproduction — Predict 1
If submission saves a job and the process crashes before enqueue, what should status show? Predict whether it is silently lost or recoverable by reconciliation.


In [ ]:
@dataclass
class JobStore:
    jobs: dict = field(default_factory=dict)
    effects: dict = field(default_factory=dict)
    next_id: int = 1
    def create(self, owner, payload):
        job_id = f'job-{self.next_id}'; self.next_id += 1
        self.jobs[job_id] = {'owner': owner, 'payload': payload, 'state': 'submitted', 'attempts': 0}
        return job_id
store = JobStore(); queue = []
job_id = store.create('ana', {'kind':'summary'})
assert store.jobs[job_id]['state'] == 'submitted' and not queue
print('Accepted durable intention exists even though queue publication is missing')


The job is not silently safe yet: an outbox/reconciliation pass must find it. **Pre-edit hypothesis:** publishing every submitted job absent from the queue makes the crash window recoverable.


## Predict 2
A `202 Accepted` response means the request was accepted. Does it mean the report is already completed? No: completion is a later state.


In [ ]:
def submit(store, queue, owner, payload, crash=False):
    jid = store.create(owner, payload)
    if not crash: queue.append(jid)
    return {'status': 202, 'job_id': jid}
accepted = submit(store, queue, 'ben', {'kind':'summary'})
assert accepted['status'] == 202 and store.jobs[accepted['job_id']]['state'] == 'submitted'
assert store.jobs[accepted['job_id']]['job_id'] if False else True
print('Accepted is not completed')


The odd-looking final assertion is intentionally harmless starter code: the real contract is the state assertion above.


## Predict 3
If a worker creates the effect and crashes before acknowledgement, will at-least-once delivery give the same job again? Yes. What prevents two reports? A durable idempotency record keyed by job ID.


In [ ]:
def reconcile(store, queue):
    queued = set(queue)
    for jid, job in store.jobs.items():
        if job['state'] == 'submitted' and jid not in queued:
            queue.append(jid)
def worker_once(store, queue, crash_after_effect=False):
    if not queue: return 'empty'
    jid = queue.pop(0); job = store.jobs[jid]
    if job['state'] in {'cancelled','dead'}: return 'skipped'
    job['state'] = 'running'; job['attempts'] += 1
    if jid in store.effects:
        job['state'] = 'succeeded'; return 'duplicate-safe'
    result = {'job_id': jid, 'text': 'synthetic report'}
    store.effects[jid] = result
    if crash_after_effect: return 'crashed-before-ack'
    job['state'] = 'succeeded'; return 'done'
reconcile(store, queue)
assert job_id in queue
queue.remove(job_id); queue.insert(0, job_id)
assert worker_once(store, queue) == 'done'
queue.insert(0, job_id)
assert worker_once(store, queue) == 'duplicate-safe'
assert len(store.effects) == 1 and store.jobs[job_id]['state'] == 'succeeded'


In [ ]:
# Failure-path reproduction: effect exists, acknowledgement is lost, then delivery repeats.
crash_store = JobStore(); crash_id = crash_store.create('ana', {'kind':'summary'}); crash_queue = [crash_id]
assert worker_once(crash_store, crash_queue, crash_after_effect=True) == 'crashed-before-ack'
crash_queue.insert(0, crash_id)
assert worker_once(crash_store, crash_queue) == 'duplicate-safe'
assert len(crash_store.effects) == 1
print('Worker crash window is duplicate-safe')


## Guided TODO — attempt before the reference solution
Write down the transition rule first: only queued/submitted jobs can run, attempts increment, permanent failure becomes dead, and acknowledgement follows a durable state. Then edit or extend the executable starter below in your notes. Run it only after you have predicted the retry state; the next cell is explicitly labeled as the reference solution.


In [ ]:
def classify_failure(kind):
    return 'retry' if kind == 'temporary' else 'dead'
def finish_failure(store, jid, kind, max_attempts=2):
    job = store.jobs[jid]
    if classify_failure(kind) == 'retry' and job['attempts'] < max_attempts:
        job['state'] = 'retry_wait'; return 'retry_wait'
    job['state'] = 'dead'; return 'dead'
# Guided TODO starter: run it, then explain why the limit prevents an infinite loop.
retry_store = JobStore(); retry_id = retry_store.create('ana', {'kind':'bad'})
retry_store.jobs[retry_id]['attempts'] = 1
assert finish_failure(retry_store, retry_id, 'temporary') == 'retry_wait'


In [ ]:
# Reference solution: exhaustion and cancellation are visible.
retry_store.jobs[retry_id]['attempts'] = 2
assert finish_failure(retry_store, retry_id, 'temporary') == 'dead'
cancel_store = JobStore(); cancel_id = cancel_store.create('ana', {})
cancel_store.jobs[cancel_id]['state'] = 'cancelled'
cancel_queue = [cancel_id]
assert worker_once(cancel_store, cancel_queue) == 'skipped'
print('Retry, dead-letter, and queued cancellation checks passed')


## Intentionally weak AI-style design
A tempting implementation marks a job `succeeded` before storing the result and acknowledges immediately. Why is this false confidence? A crash can expose success with no result, and redelivery can duplicate an effect. The worker above stores the effect and state before treating delivery as finished.


## Independent challenge — attempt before checking
Add an authorization check so only the owner can read status. Decide what an unknown job and another owner's job should return, write your expected cases, and then compare with the executable check below.


In [ ]:
def status(store, jid, actor):
    job = store.jobs.get(jid)
    if job is None or job['owner'] != actor: return {'status': 404}
    return {'status': 200, 'state': job['state']}
assert status(store, job_id, 'ana')['status'] == 200
assert status(store, job_id, 'ben')['status'] == 404
assert status(store, 'missing', 'ana')['status'] == 404
print('Status authorization challenge passed')


## Exit questions
1. Where is acknowledgement safe?
2. What does idempotency prove?
3. What remains unproved?

### Answers
1. After durable success or a durable terminal failure.
2. One business effect for one job key, not one worker execution.
3. Real broker durability, leases, cross-process transactions, and production throughput.

## Evidence handoff
Save state transcripts, crash-window reconciliation, duplicate-safe effect count, retry/dead/cancel results, authorization output, and your AI critique.
